# 06 · Dataset Construction

**Purpose:** Align news features with market data and compute prediction targets.

**Alignment logic:**  
Features computed from articles published on day `d` are aligned with
the stock price on day `d`.  The target is the **next trading day's return**
(`close_{d+1} - close_d) / close_d`).
This ensures no look-ahead bias: you observe news today, predict tomorrow.

**Targets:**
- `next_day_return` — continuous return (regression)
- `direction` — 1 if `next_day_return > 0`, else 0 (classification)

**Input:** `features.parquet`, `stock_prices.parquet`  
**Output:** `modeling_dataset.parquet`

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

NB_DIR = Path().resolve()
sys.path.insert(0, str(NB_DIR))
from utils import load, save

In [ ]:
features     = load("features")
stock_prices = load("stock_prices")

print(f"Feature rows:    {len(features):,}")
print(f"Price rows:      {len(stock_prices):,}")

## 1 · Compute returns and targets

In [ ]:
def compute_targets(prices: pd.DataFrame) -> pd.DataFrame:
    df = prices.sort_values(["ticker", "date"]).copy()
    df["date"] = pd.to_datetime(df["date"])

    df["daily_return"]    = df.groupby("ticker")["close"].pct_change()
    df["next_day_return"] = df.groupby("ticker")["daily_return"].shift(-1)
    df["direction"]       = (df["next_day_return"] > 0).astype("Int64")

    return df.dropna(subset=["next_day_return"])


prices = compute_targets(stock_prices)
print(f"Price rows with targets: {len(prices):,}")
print(f"Direction balance: {prices['direction'].mean():.2%} up days")

## 2 · Merge features with prices

In [ ]:
features["date"] = pd.to_datetime(features["date"])

dataset = prices.merge(features, on=["ticker", "date"], how="left")

# Rows with no news signal on this day → fill with zero (no-news = neutral)
feature_cols = [c for c in features.columns if c not in ["ticker", "date"]]
dataset[feature_cols] = dataset[feature_cols].fillna(0)

news_days = (dataset["news_count"] > 0).sum()
print(f"Dataset shape:          {dataset.shape}")
print(f"Tickers:                {dataset['ticker'].nunique()}")
print(f"Date range:             {dataset['date'].min().date()} → {dataset['date'].max().date()}")
print(f"Rows with news signal:  {news_days:,} / {len(dataset):,} ({news_days/len(dataset):.1%})")

## 3 · Return distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

clipped = dataset["next_day_return"].clip(-0.1, 0.1)
clipped.hist(bins=60, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Next-day return distribution (clipped at ±10%)")
axes[0].set_xlabel("Return")

# Avg sentiment vs avg next-day return (decile buckets)
no_zero = dataset[dataset["news_count"] > 0].copy()
no_zero["sent_decile"] = pd.qcut(no_zero["avg_sentiment"], q=10, labels=False, duplicates="drop")
bucket = no_zero.groupby("sent_decile")["next_day_return"].mean()
bucket.plot.bar(ax=axes[1], color="steelblue")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Avg next-day return by sentiment decile")
axes[1].set_xlabel("Sentiment decile (0=most negative)")

plt.tight_layout()
plt.show()

## 4 · Coverage summary per ticker

In [ ]:
summary = dataset.groupby("ticker").agg(
    total_days   =("date", "count"),
    news_days    =("news_count", lambda x: (x > 0).sum()),
    avg_return   =("next_day_return", "mean"),
    std_return   =("next_day_return", "std"),
).assign(news_coverage=lambda d: (d["news_days"] / d["total_days"]).round(3))

print(summary.sort_values("news_days", ascending=False).head(20).to_string())

In [ ]:
save(dataset, "modeling_dataset")